In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re

* Connect to weather API, get output into a table
* One table per university with all available data then combine to comprehensive u-weather table
    * Start easy then get more challenging
* One table per university with u-data (e.g., # of students), combine to comprehensive u-data table
    * Analyze data, define "severe" weather circumstances, apply to next summary table
* Combine summary statistics from u-data and u-weather to final table to answer hmwk questions.

In [ ]:
# Create dictionary of universities and lat/long
places = {
    'u_miss': {'latitude': 34.365, 'longitude': -89.5383},
    'tulane': {'latitude': 29.9401, 'longitude': -90.1207},
    'g_tech': {'latitude': 33.7756, 'longitude': -84.3963},
    'u_ark': {'latitude': 36.0692, 'longitude': -94.1756},
    'u_bama': {'latitude': 33.2113, 'longitude': -87.5398}
}

# Access lat/long values within Places dictionary
lats_longs = places.values()
# print(lats_longs)

# Bring all lats/longs into a comma-separated list of strings
all_lats = ','.join(str(d['latitude']) for d in lats_longs)
all_longs = ','.join(str(d['longitude']) for d in lats_longs)

# Add base url for API
base_url = "https://archive-api.open-meteo.com/v1/archive"

# Add parameters into a dictionary
api_params = {
    'latitude': all_lats,
    'longitude': all_longs,
    'start_date': '2026-01-01',
    'end_date': '2026-01-31',
    'daily': ['precipitation_sum', 'snowfall_sum', 'precipitation_hours', 'temperature_2m_max', 'temperature_2m_mean', 'temperature_2m_min'],
    'timezone': 'auto',
    'temperature_unit': 'fahrenheit',
    'wind_speed_unit': 'mph',
    'precipitation_unit': 'inch'
}

# Get response
response = requests.get(base_url, params=api_params)
data = response.json()
## Preview the data
# data

In [ ]:
# Use pandas to bring data into a data frame
weather_df = pd.DataFrame(data)

# Create a list of university names
school_names = list(places.keys())

# Add series of names to weather_df
weather_df['school'] = school_names

# # Check the results
# weather_df

In [ ]:
# Instantiate list of dataframes
all_frames = []

for index, row in weather_df.iterrows():
    # Get contents of daily dict into a temp dataframe
    tempdf = pd.DataFrame(row['daily'])
    # Add school name to df
    tempdf['school'] = row['school']
    # Append to final dataframe
    all_frames.append(tempdf)

# Concatenate list of 5 frames into one big frame
final_df = pd.concat(all_frames, ignore_index=True)

# # Review data
# print(final_df)

In [ ]:
final_df.head()

In [ ]:
def fetch_enrollment(place_key, url, search_word, regex_pattern, places_dict):
    """
    Scrapes a specific URL for enrollment numbers and updates the places dictionary.
    
    Args:
        place_key (str): The dictionary key (e.g., 'u_miss').
        url (str): The website to scrape.
        search_word (str): A unique word to find the right paragraph (e.g., 'welcomed').
        regex_pattern (str): The raw string pattern to extract the number.
        places_dict (dict): The master dictionary to update.
    """
    
    # Standard headers to look like a browser
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    try:
        response = requests.get(url, headers=headers)

        # If initial call doesn't work, hop out of the function
        if response.status_code != 200:
            print(f"[{place_key}] Failed: Status Code {response.status_code}")
            return

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Finds the text string *anywhere*, then grabs the parent tag (p, li, div, etc.)
        target_element = soup.find(string=re.compile(search_word))
        target_par = target_element.parent if target_element else None
        
        if not target_par:
            print(f"[{place_key}] Connected, but couldn't find a paragraph containing '{search_word}'.")
            return

        # 2. Extract text and run regex
        target_content = target_par.get_text()
        match = re.search(regex_pattern, target_content)
        
        if match:
            # Clean the number (remove commas)
            raw_number = match.group(1)
            enrollment_count = int(raw_number.replace(',', ''))
            
            # Update the dictionary
            places_dict[place_key]['total_pop'] = enrollment_count
            print(f"[{place_key}] Success! Enrollment: {enrollment_count}")
        else:
            print(f"[{place_key}] Found paragraph, but regex didn't match. Content snippet: '{target_content[:50]}...'")

    except Exception as e:
        print(f"[{place_key}] Error: {e}")

In [ ]:
# Use dictionary to compile sources to scrape so I can iterate over them with the function
scraping_config = {
    # University of Mississippi
    'u_miss': {
        'url': 'https://olemiss.edu/news/2025/11/um-reaches-record-enrollment-for-third-straight-year/index.html',
        'search_word': 'welcomed', 
        'regex_pattern': r'welcomed\s+([\d,]+)\s+students' 
    },

    # Trouble with the iframe--can't do simple HTML extraction
    # Tulane University
    'tulane': {
        'url': 'https://oair.tulane.edu/university-enrollment-reports', 
        'search_word': 'TBD', 
        'regex_pattern': 'TBD'
    },
    
    # Georgia Tech
    'g_tech': {
        'url': 'https://www.gatech.edu/news/2025/04/22/georgia-tech-reports-strong-enrollment-growth-roi',
        'search_word': 'total enrollment',
        'regex_pattern': r'total enrollment is\s+([\d,]+)\s+students'
    },
    
    # University of Arkansas
    'u_ark': {
        'url': 'https://www.uark.edu/about/quick-facts.php',
        'search_word': 'Record student enrollment',
        'regex_pattern': r'([\d,]+)\s+Record student enrollment'
    },
    
    # University of Alabama
    'u_bama': {
        'url': 'https://www.ua.edu/about/quick-facts/',
        'search_word': 'welcoming',
        'regex_pattern': r'welcoming\s+([\d,]+)\s+students'
    }
}

In [ ]:
# Loop through scraping dictionary to get each school and its attributes, 

print("Starting Scraping Job...")
print("-" * 30)

for school_key, config in scraping_config.items():

    # Used TBD as placeholder for schools as they were added, kept.
    if config['url'] != 'TBD':
        print(f"Scraping {school_key}...")
        
        fetch_enrollment(
            place_key=school_key,
            url=config['url'],
            search_word=config['search_word'], 
            regex_pattern=config['regex_pattern'],
            places_dict=places  # Pass your master dictionary to be updated
        )
    else:
        print(f"Skipping {school_key} (No URL configured)")

print("-" * 30)
print("Job Complete. Updated Data:")
print(places)